# Satellite-Based Chlorophyll and NDCI Extraction for Lake Monitoring

This notebook extracts chlorophyll-a concentration estimates and Normalized Difference Chlorophyll Index (NDCI) values from multiple satellite platforms (MODIS-Terra, MODIS-Aqua, and Sentinel-2) for specified lake locations.

## Overview

- Purpose: Generate time series of chlorophyll indices from satellite imagery
- Study Areas: Detroit Lake and Upper Klamath Lake
- Satellite Sensors: MODIS (Terra and Aqua) and Sentinel-2 (2A and 2B).
- Output: CSV files with date-stamped chlorophyll index (e.g., NDCI) values

In [2]:
"""
Initialize Google Earth Engine (GEE) connection and authentication.

This cell sets up the GEE Python API for accessing satellite imagery collections.
Authentication is required on first use or when credentials expire.
"""

import ee
import math
import pandas as pd

# Authenticate and initialize GEE with your registered project
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')  # Replace with your GEE Cloud Project ID

## MODIS Chlorohpyll Extraction (MODIS-Terra and MODIS-Aqua)

### Overview

MODIS (Moderate Resolution Imaging Spectroradiometer) provides daily global coverage in the visible bands at 250, 500, and 1,000 m resolution from two satellites: Terra (morning overpass) and Aqua (afternoon overpass). This section derives chlorophyll-a concentrations using an empirical green-to-red ratio algorithm using the 500 m bands.

In [3]:
# ============================================================================
# MODIS Chlorophyll-a Extraction (Revised: QA mask + water mask + ROI median)
# ============================================================================

# -----------------------------
# User parameters
# -----------------------------

# Lakes and output file names
lakes = [
    dict(
        name='Detroit',
        lon=-122.184, lat=44.711,
        terra_export='Detroit_MODIS_Terra_500m_Chl_ROI',
        aqua_export='Detroit_MODIS_Aqua_500m_Chl_ROI',
        # Optional per-lake coefficients; if None, fall back to defaults below
        a=None, b=None
    ),
    dict(
        name='UpperKlamath',
        lon=-121.900, lat=42.400,
        terra_export='Klamath_MODIS_Terra_500m_Chl_ROI',
        aqua_export='Klamath_MODIS_Aqua_500m_Chl_ROI',
        a=None, b=None
    ),
]

# Analysis window
start_date = '2011-01-01'
end_date   = '2025-12-31'

# Retrieval coefficients
#****************************************************
# [TODO] VERIFY AND UPDATE
#****************************************************
DEFAULT_A = 1.70  # slope
DEFAULT_B = 1.54  # intercept

# ROI and masking controls
ROI_RADIUS_M          = 1000   # radius for offshore sampling ROI (meters)
SHORELINE_BUFFER_M    = 500    # erode water mask away from land by this distance (meters)
WATER_OCC_THRESHOLD   = 75     # JRC Global Surface Water "occurrence" threshold (0-100)
GSW_DATASET_ID        = 'JRC/GSW1_4/GlobalSurfaceWater'  # if unavailable, change to 'JRC/GSW1_3/GlobalSurfaceWater'

# MODIS collections (Surface Reflectance, 500 m)
TERRA_COL_ID = 'MODIS/061/MOD09GA'
AQUA_COL_ID  = 'MODIS/061/MYD09GA'

# Bands for green/red ratio (500 m SR)
B_GREEN = 'sur_refl_b04'  # ~555 nm
B_RED   = 'sur_refl_b01'  # ~645 nm
SR_SCALE = 1e-4           # scale factor for MOD09/MYD09 SR bands

# -----------------------------
# Masks and helpers
# -----------------------------

def mask_mod09(img):
    """
    Apply MOD09/MYD09 QA-based mask using state_1km:
      - Cloud state (bits 0-1): mask cloudy or mixed
      - Cloud shadow (bit 2)
      - Cirrus (bits 8-9)
      - Internal cloud (bit 10)
      - Snow/ice (bit 12)
    """
    qa = img.select('state_1km')
    cloud_state = qa.bitwiseAnd(3)                        # bits 0-1
    is_cloudy_or_mixed = cloud_state.eq(1).Or(cloud_state.eq(2))
    shadow   = qa.bitwiseAnd(1 << 2).neq(0)               # bit 2
    cirrus   = qa.bitwiseAnd(3 << 8).neq(0)               # bits 8-9
    intcloud = qa.bitwiseAnd(1 << 10).neq(0)              # bit 10
    snowice  = qa.bitwiseAnd(1 << 12).neq(0)              # bit 12

    mask = (is_cloudy_or_mixed
            .Or(shadow)
            .Or(cirrus)
            .Or(intcloud)
            .Or(snowice)
            ).Not()

    return img.updateMask(mask)

def build_water_mask():
    """
    Build a persistent open-water mask (30 m) from JRC Global Surface Water 'occurrence'.
    Erode by SHORELINE_BUFFER_M to reduce adjacency (shoreline) contamination.
    """
    gsw = ee.Image(GSW_DATASET_ID).select('occurrence')
    water = gsw.gte(WATER_OCC_THRESHOLD)
    # Erode water by shoreline buffer (distance in meters)
    water_eroded = water.focal_min(radius=SHORELINE_BUFFER_M, units='meters')
    return water_eroded

WATER_MASK = build_water_mask()

def add_chlorophyll(img, a, b):
    """
    Add chlorophyll-a band using a log-ratio model:
      log10(Chl) = a * log10( (SR555) / (SR645) ) + b
    Notes:
      - We use SR directly; for a ratio, dividing by π to form Rrs would cancel.
      - Mask out non-positive reflectances before ratio/log10.
    """
    g = img.select(B_GREEN).multiply(SR_SCALE)
    r = img.select(B_RED).multiply(SR_SCALE)
    valid = g.gt(0).And(r.gt(0))
    ratio = g.divide(r).updateMask(valid)
    log_ratio = ratio.log10()
    log10_chl = log_ratio.multiply(a).add(b)
    chl = ee.Image(10).pow(log10_chl).rename('chlor_a')  # µg/L
    return img.addBands(chl)

def imagecollection_to_features(col_id, roi_geom, sensor_tag, a, b):
    """
    Build a per-image feature collection with median Chl-a over the ROI.
    Applies:
      - QA mask
      - Persistent water mask + shoreline erosion
      - ROI median at 500 m
    """
    def per_image(img):
        img = mask_mod09(img)
        img = img.updateMask(WATER_MASK)              # keep only persistent open water
        img = add_chlorophyll(img, a, b)

        stats = img.select('chlor_a').reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=roi_geom,
            scale=500,
            maxPixels=1e9,
            bestEffort=True
        )

        return ee.Feature(None, {
            'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'chl':  stats.get('chlor_a'),
            'sensor': sensor_tag
        })

    fc = (ee.ImageCollection(col_id)
          .filterDate(start_date, end_date)
          .filterBounds(roi_geom)
          .map(per_image)
          .filter(ee.Filter.notNull(['chl'])))

    return fc

# -----------------------------
# Main processing loop
# -----------------------------

for lake in lakes:
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi = center.buffer(ROI_RADIUS_M)

    # Per-lake coefficients (fallback to defaults if not provided)
    a = lake.get('a', None)
    b = lake.get('b', None)
    if a is None: a = DEFAULT_A
    if b is None: b = DEFAULT_B

    # Build time series for Terra and Aqua
    terra_fc = imagecollection_to_features(TERRA_COL_ID, roi, 'Terra', a, b)
    aqua_fc  = imagecollection_to_features(AQUA_COL_ID,  roi, 'Aqua',  a, b)

    # Quick availability prints
    print(f"{lake['name']} Terra rows =", terra_fc.size().getInfo())
    print(f"{lake['name']} Aqua  rows =", aqua_fc.size().getInfo())

    # Client-side CSV export

    # Terra
    terra_rows = terra_fc.getInfo()['features']
    terra_records = [f['properties'] for f in terra_rows]
    terra_df = pd.DataFrame.from_records(terra_records)
    terra_df.to_csv(lake['terra_export'] + '.csv', index=False)

    # Aqua
    aqua_rows = aqua_fc.getInfo()['features']
    aqua_records = [f['properties'] for f in aqua_rows]
    aqua_df = pd.DataFrame.from_records(aqua_records)
    aqua_df.to_csv(lake['aqua_export'] + '.csv', index=False)

    # (Optional) Also enable server-side exports if desired:
    # ee.batch.Export.table.toDrive(
    #     collection=terra_fc, description=lake['terra_export'], fileFormat='CSV'
    # ).start()
    # ee.batch.Export.table.toDrive(
    #     collection=aqua_fc, description=lake['aqua_export'], fileFormat='CSV'
    # ).start()

print("Done!")


Detroit Terra rows = 1928
Detroit Aqua  rows = 1999
UpperKlamath Terra rows = 2495
UpperKlamath Aqua  rows = 2401
Done!


## Sentinel-2 Processing (10m resolution)

### Overview

Sentinel-2 provides high spatial (10-20m) and temporal (5-day revisit) resolution multispectral imagery since 2015. The constellation consists of two satellites (2A and 2B) offering 13 spectral bands optimized for vegetation and water monitoring.


In [4]:
# Sentinel-2 NDCI Extraction (following Johansen et al., 2024)

import ee
import pandas as pd

# -----------------------------------------------------------------------------
# Define lake locations and output filenames
# -----------------------------------------------------------------------------
lakes = [
    dict(name='Detroit',
         lon=-122.184, lat=44.711,
         export_id='Detroit_S2_NDCI_500m'),
    dict(name='UpperKlamath',
         lon=-121.900, lat=42.400,
         export_id='Klamath_S2_NDCI_500m')
]

# Temporal range (Sentinel-2 available from July 2015)
start_date, end_date = '2015-07-01', '2025-12-31'

# Spatial buffer for 500 m x 500 m Region of Interest (ROI)
half_size_m = 250  # metres

# Sentinel-2 Level-2A Surface Reflectance (harmonized)
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

# -----------------------------------------------------------------------------
# Preprocessing & Index Functions
# -----------------------------------------------------------------------------

def mask_s2_scl_water(img):
    """
    Keep only SCL water (class 6), following Johansen et al. (2024), eroded
    by 1 pixel to reduce shoreline adjacency, and remove edge artifacts
    where B8A == 0.
    """
    scl = img.select('SCL')
    water = scl.eq(6)
    # Erode 1 pixel (20 m) to avoid shoreline bleed
    water_eroded = water.focal_min(1)
    edge_ok = img.select('B8A').gt(0)
    mask = water_eroded.And(edge_ok)
    return img.updateMask(mask)

def add_ndci(img):
    """
    NDCI = (B5 - B4) / (B5 + B4)
      - B5: 705 nm (20 m)
      - B4: 665 nm (10 m) -> resample & reproject to B5 grid
    Cast to float to avoid integer division; NDCI is scale-invariant.
    """
    b5 = img.select('B5').toFloat()  # 20 m native
    # Resample B4 to match B5's 20 m grid and projection
    b4 = (img.select('B4')
            .toFloat()
            .resample('bilinear')
            .reproject(b5.projection()))
    ndci = b5.subtract(b4).divide(b5.add(b4)).rename('NDCI')
    return img.addBands(ndci)

def preprocess_s2(img):
    """
    Preprocessing follows Johansen et al., 2024:
      1) SCL water-only (class 6) with 1 px erosion
      2) Edge artifact removal via B8A > 0
    """
    return mask_s2_scl_water(img)

def img_to_feature(img, roi, tag):
    """
    Convert image to a Feature with mean NDCI over ROI (20 m).
    Returns a Feature even if NDCI is null. We will drop nulls later.
    """
    mean_ndci = img.select('NDCI').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=20,             # match red-edge native resolution
        maxPixels=1e9,
        bestEffort=True
    ).get('NDCI')

    props = {
        'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
        'ndci': mean_ndci,
        'sensor': tag
    }
    return ee.Feature(None, props)

# -----------------------------------------------------------------------------
# Process each lake
# -----------------------------------------------------------------------------
for lake in lakes:
    # Define ROI: 500 m x 500 m box centered at given coordinate
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi = center.buffer(half_size_m).bounds()

    tag = f"{lake['name']}_S2"

    # Build collection --> preprocess --> add NDCI
    processed = (s2
        .filterDate(start_date, end_date)
        .filterBounds(roi)
        .map(preprocess_s2)
        .map(add_ndci))

    # Map images to features (per-scene ROI mean), then drop null NDCI values
    fc = ee.FeatureCollection(
        processed.map(lambda img: img_to_feature(img, roi, tag))
    ).filter(ee.Filter.notNull(['ndci']))

    # Print scene count
    count = fc.size().getInfo()
    print(lake['name'], 'valid Sentinel-2 scenes =', count)

    # Client-side CSV export
    rows = fc.getInfo()['features']
    records = [f['properties'] for f in rows]
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['export_id'] + '.csv', index=False)

print("Done!")


Detroit valid Sentinel-2 scenes = 265
UpperKlamath valid Sentinel-2 scenes = 1272
Done!
